<a href="https://colab.research.google.com/github/albertiniufu/gbc063/blob/main/atividades/0_qlearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q-Learning com FrozenLake-v1 ⛄ e Taxi-v4 🚕

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/thumbnail.jpg" alt="Unit 2 Thumbnail">

Neste notebook, **você programará seu primeiro agente de Aprendizagem por Reforço do zero** para jogar FrozenLake ❄️ usando Q-Learning e experimentar diferentes configurações.

Este notebook é uma simplificação do unit2.ipynb do **Curso de Aprendizagem por Reforço Profundo** (Deep Reinforcement Learning) https://huggingface.co/datasets/huggingface-deep-rl-course/.

⬇️ Aqui está um exemplo do que **você alcançará em apenas alguns minutos.** ⬇️


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/envs.gif" alt="Environments"/>

### 🎮 Ambientes:

- [FrozenLake-v1](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
- [Taxi-v4](https://gymnasium.farama.org/environments/toy_text/taxi/)

### 📚 Biblioteca de RL:

- Python e NumPy
- [Gymnasium](https://gymnasium.farama.org/)


## Objetivos deste notebook 🏆

No final do notebook, você irá:

- Saber usar o **Gymnasium**, a biblioteca de ambientes.
- Saber programar um agente de Q-Learning do zero.


## Um breve resumo sobre Q-Learning

*Q-Learning* **é o algoritmo de RL que**:

- Treina a *Q-Function* (Função Q), uma **função de valor de ação** codificada na memória interna por uma *Q-table* (tabela Q) **que contém todos os valores de pares estado-ação.**

- Dado um estado e uma ação, nossa Função Q **pesquisará na Q-table pelo valor correspondente.**
    
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-function-2.jpg" alt="Q function"  width="100%"/>

- Quando o treinamento for concluído, **teremos uma Q-Function ótima e, portanto, uma Q-Table ótima.**
    
- E se **tivermos uma Q-function ótima**, temos uma política ótima, já que **sabemos, para cada estado, a melhor ação a ser tomada.**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/link-value-policy.jpg" alt="Link value policy"  width="100%"/>


Mas, no início, nossa **Q-Table é inútil, pois fornece valores arbitrários para cada par estado-ação (na maioria das vezes inicializamos a Q-Table com valores 0)**. No entanto, à medida que exploramos o ambiente e atualizamos nossa Q-Table, ela nos fornecerá aproximações cada vez melhores

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/q-learning.jpeg" alt="q-learning.jpeg" width="100%"/>

Este é o pseudocódigo do Q-Learning:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>


# Vamos programar nosso primeiro algoritmo de Aprendizagem por Reforço 🚀

## Instalar dependências e criar uma tela virtual 🔽

No notebook, precisaremos gerar um vídeo de replay. Para fazer isso no Colab, **precisamos ter uma tela virtual para renderizar o ambiente** (e, assim, gravar os quadros/frames).

Portanto, a célula a seguir instalará as bibliotecas, criará e executará uma tela virtual 🖥

Instalaremos várias:

- `gymnasium`: Contém os ambientes FrozenLake-v1 ⛄ e Taxi-v4 🚕.
- `pygame`: Usado para a interface de usuário (UI) do FrozenLake-v1 e do Taxi-v4.
- `numpy`: Usado para manipular nossa Q-table.


In [ ]:
!pip install -r https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit2/requirements-unit2.txt

In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg xvfb
!pip3 install pyvirtualdisplay

Para garantir que as novas bibliotecas instaladas sejam usadas, **às vezes é necessário reiniciar o ambiente de execução (runtime) do notebook**. A próxima célula forçará o **runtime a falhar, então você precisará se conectar novamente e executar o código a partir daqui**. Graças a esse truque, **poderemos executar nossa tela virtual.**

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# Tela virtual
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## Importar os pacotes 📦

Além das bibliotecas instaladas, também usamos:

- `random`: Para gerar números aleatórios (o que será útil para a política épsilon-greedy).
- `imageio`: Para gerar um vídeo de replay.

In [ ]:
import numpy as np
import gymnasium as gym
import random
import imageio
import os
import tqdm

import pickle
from tqdm.notebook import tqdm

Agora estamos prontos para programar nosso algoritmo de Q-Learning 🔥

# Parte 1: Frozen Lake ⛄ (versão não escorregadia / non-slippery)

## Criar e entender o [ambiente FrozenLake ⛄](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
---

💡 Um bom hábito ao começar a usar um ambiente é verificar sua documentação

👉 https://gymnasium.farama.org/environments/toy_text/frozen_lake/

---

Vamos treinar nosso agente de Q-Learning **para navegar do estado inicial (S) até o estado objetivo (G) andando apenas em pisos congelados (F) e evitando buracos (H)**.

Podemos ter dois tamanhos de ambiente:

- `map_name="4x4"`: uma versão de grade 4x4
- `map_name="8x8"`: uma versão de grade 8x8


O ambiente possui dois modos:

- `is_slippery=False`: O agente sempre se move **na direção pretendida** devido à natureza não escorregadia do lago congelado (determinístico).
- `is_slippery=True`: O agente **pode nem sempre se mover na direção pretendida** devido à natureza escorregadia do lago congelado (estocástico).

Por enquanto, vamos manter as coisas simples com o mapa 4x4 e não escorregadio.
Adicionamos um parâmetro chamado `render_mode` que especifica como o ambiente deve ser visualizado. No nosso caso, porque **queremos gravar um vídeo do ambiente no final, precisamos definir render_mode como rgb_array**.

Como [explicado na documentação](https://gymnasium.farama.org/api/env/#gymnasium.Env.render) “rgb_array”: Retorna um único quadro (frame) representando o estado atual do ambiente. Um quadro é um np.ndarray com formato (x, y, 3) representando valores RGB para uma imagem de x por y pixels.

In [ ]:
# Crie o ambiente FrozenLake-v1 usando mapa 4x4, versão não escorregadia e render_mode="rgb_array"
env = gym.make() # TODO use os parâmetros corretos

### Solução (só abra depois de tentar, ler documentação e tentar de novo 😜)

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

Você pode criar sua própria grade personalizada assim:

```python
desc=["SFFF", "FHFH", "FFFH", "HFFG"]
gym.make('FrozenLake-v1', desc=desc, is_slippery=True)
```

mas usaremos o ambiente padrão por enquanto.

### Vamos ver como é o Ambiente:


In [ ]:
# Criamos nosso ambiente com gym.make("<nome_do_ambiente>")- `is_slippery=False`: O agente sempre se move na direção pretendida devido à natureza não escorregadia do lago congelado (determinístico).
print("_____ESPAÇO DE OBSERVAÇÃO_____ \n")
print("Espaço de Observação", env.observation_space)
print("Amostra de observação", env.observation_space.sample()) # Obtém uma observação aleatória

Vemos com `Observation Space Shape Discrete(16)` que a observação é um número inteiro representando a **posição atual do agente como current_row * ncols + current_col (onde tanto a linha quanto a coluna começam em 0)**.

Por exemplo, a posição do objetivo (goal) no mapa 4x4 pode ser calculada da seguinte forma: 3 * 4 + 3 = 15. O número de observações possíveis depende do tamanho do mapa. **Por exemplo, o mapa 4x4 tem 16 observações possíveis.**


Por exemplo, é assim que state = 0 se parece:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/frozenlake.png" alt="FrozenLake">

In [ ]:
print("\n _____ESPAÇO DE AÇÃO_____ \n")
print("Formato do Espaço de Ação", env.action_space.n)
print("Amostra do Espaço de Ação", env.action_space.sample()) # Realiza uma ação aleatória

O espaço de ação (o conjunto de ações possíveis que o agente pode tomar) é discreto com 4 ações disponíveis 🎮:
- 0: IR PARA A ESQUERDA (GO LEFT)
- 1: IR PARA BAIXO (GO DOWN)
- 2: IR PARA A DIREITA (GO RIGHT)
- 3: IR PARA CIMA (GO UP)

Função de recompensa 💰:
- Alcançar o objetivo: +1
- Cair no buraco: 0
- Alcançar o piso congelado: 0

## Criar e Inicializar a Q-table 🗄️

(👀 Passo 1 do pseudocódigo)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>

É hora de inicializar nossa Q-table! Para saber quantas linhas (estados) e colunas (ações) usar, precisamos saber o espaço de ação e de observação. Já sabemos seus valores anteriores, mas queremos obtê-los programaticamente para que nosso algoritmo se generalize para diferentes ambientes. O Gym nos fornece uma maneira de fazer isso: `env.action_space.n` e `env.observation_space.n`


In [ ]:
state_space =
print("Existem ", state_space, " estados possíveis")

action_space =
print("Existem ", action_space, " ações possíveis")

In [ ]:
# Vamos criar nossa Qtable de tamanho (state_space, action_space) e inicializar cada valor em 0 usando np.zeros. np.zeros precisa de uma tupla (a,b)
def initialize_q_table(state_space, action_space):
  Qtable =
  return Qtable

In [ ]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)

### Solução

In [ ]:
state_space = env.observation_space.n
print("Existem ", state_space, " estados possíveis")

action_space = env.action_space.n
print("Existem ", action_space, " ações possíveis")

In [ ]:
# Vamos criar nossa Qtable de tamanho (state_space, action_space) e inicializar cada valor em 0 usando np.zeros
def initialize_q_table(state_space, action_space):
  Qtable = np.zeros((state_space, action_space))
  return Qtable

In [ ]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)

## Definir a política gulosa (greedy policy) 🤖

Lembre-se de que temos duas políticas, pois o Q-Learning é um algoritmo **off-policy**. Isso significa que estamos usando uma **política diferente para agir e para atualizar a função de valor**.

- Política épsilon-greedy (política de ação)
- Política gulosa / greedy (política de atualização)

A política gulosa também será a política final que teremos quando o agente de Q-Learning concluir o treinamento. A política gulosa é usada para selecionar uma ação usando a Q-table.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/off-on-4.jpg" alt="Q-Learning" width="100%"/>


In [ ]:
def greedy_policy(Qtable, state):
  # Exploração/Exploitation: pegar a ação com o maior valor de estado-ação
  action =

  return action

#### Solução

In [ ]:
def greedy_policy(Qtable, state):
  # Exploitation: pegar a ação com o maior valor de estado-ação
  action = np.argmax(Qtable[state][:])

  return action

## Definir a política épsilon-greedy 🤖

Épsilon-greedy é a política de treinamento que lida com o trade-off entre exploração e aproveitamento (exploration/exploitation).

A ideia com o épsilon-greedy:

- Com *probabilidade 1 - ɛ* : **fazemos exploitation** (ou seja, nosso agente seleciona a ação com o maior valor de par estado-ação).

- Com *probabilidade ɛ*: fazemos **exploração** (tentando uma ação aleatória).

Conforme o treinamento continua, **reduzimos progressivamente o valor de epsilon, pois precisaremos de cada vez menos exploração e mais exploitation.**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-4.jpg" alt="Q-Learning" width="100%"/>


In [ ]:
def epsilon_greedy_policy(Qtable, state, epsilon):
  rnd = np.random.default_rng()
  # Gerar aleatoriamente um número entre 0 e 1
  random_num = rnd.random()
  # se random_num > maior que epsilon --> exploitation
  if random_num > epsilon:
    # Pegar a ação com o maior valor dado um estado
    # np.argmax pode ser útil aqui
    action = greedy_policy(Qtable, state)
  # senão --> exploração
  else:
    action = env.action_sample.sample() # Realizar uma ação aleatória

  return action

#### Solução

In [ ]:
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Gerar aleatoriamente um número entre 0 e 1
  random_num = random.uniform(0,1)
  # se random_num > maior que epsilon --> exploitation
  if random_num > epsilon:
    # Pegar a ação com o maior valor dado um estado
    # np.argmax pode ser útil aqui
    action = greedy_policy(Qtable, state)
  # senão --> exploração
  else:
    action = env.action_space.sample()

  return action

## Definir os hiperparâmetros ⚙️

Os hiperparâmetros relacionados à exploração são alguns dos mais importantes.

- Precisamos garantir que nosso agente **explore o suficiente do espaço de estados** para aprender uma boa aproximação de valor. Para fazer isso, precisamos ter um decaimento progressivo do epsilon.
- Se você diminuir o epsilon muito rápido (taxa de decaimento / decay_rate muito alta), **você corre o risco de seu agente ficar preso**, já que ele não explorou o suficiente do espaço de estados e, portanto, não consegue resolver o problema.

In [ ]:
# Parâmetros de treinamento
n_training_episodes = 10000  # Total de episódios de treinamento
learning_rate = 0.7          # Taxa de aprendizado (learning rate)

# Parâmetros de avaliação
n_eval_episodes = 100        # Número total de episódios de teste

# Parâmetros do ambiente
env_id = "FrozenLake-v1"     # Nome do ambiente
max_steps = 99               # Máximo de passos por episódio
gamma = 0.95                 # Taxa de desconto (discounting rate)
eval_seed = []               # A semente (seed) de avaliação do ambiente

# Parâmetros de exploração
max_epsilon = 1.0             # Probabilidade de exploração no início
min_epsilon = 0.05            # Probabilidade mínima de exploração
decay_rate = 0.0005            # Taxa de decaimento exponencial para a probabilidade de exploração

## Criar o método de loop de treinamento

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>

O loop de treinamento funciona assim:

```
Para cada episódio no total de episódios de treinamento:

Reduzir o epsilon (já que precisamos de cada vez menos exploração)
Reiniciar o ambiente

  Para cada passo em max timesteps:    
    Escolher a ação At usando a política épsilon-greedy
    Tomar a ação (a) e observar o estado resultante (s') e a recompensa (r)
    Atualizar o Q-value Q(s,a) usando a equação de Bellman Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
    Se finalizado (done), terminar o episódio
    Nosso próximo estado é o novo estado
```

In [ ]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reduzir epsilon (porque precisamos de cada vez menos exploração)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reiniciar o ambiente
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # repetir
    for step in range(max_steps):
      # Escolher a ação At usando a política épsilon-greedy
      action =

      # Tomar a ação At e observar Rt+1 e St+1
      # Tomar a ação (a) e observar o estado resultante (s') e a recompensa (r)
      new_state, reward, terminated, truncated, info =

      # Atualizar Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      Qtable[state][action] =

      # Se terminado ou truncado, encerrar o episódio
      if terminated or truncated:
        break

      # Nosso próximo estado é o novo estado
      state = new_state
  return Qtable

#### Solução

In [ ]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reduzir epsilon (porque precisamos de cada vez menos exploração)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reiniciar o ambiente
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # repetir
    for step in range(max_steps):
      # Escolher a ação At usando a política épsilon-greedy
      action = epsilon_greedy_policy(Qtable, state, epsilon)

      # Tomar a ação At e observar Rt+1 e St+1
      # Tomar a ação (a) e observar o estado resultante (s') e a recompensa (r)
      new_state, reward, terminated, truncated, info = env.step(action)

      # Atualizar Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      Qtable[state][action] = Qtable[state][action] + learning_rate * (reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action])

      # Se terminado ou truncado, encerrar o episódio
      if terminated or truncated:
        break

      # Nosso próximo estado é o novo estado
      state = new_state
  return Qtable

## Treinar o agente de Q-Learning 🏃

In [ ]:
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)

## Vamos ver como nossa tabela de Q-Learning está agora 👀

In [ ]:
Qtable_frozenlake

## O método de avaliação 📝

- Definimos o método de avaliação que usaremos para testar nosso agente de Q-Learning.

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
  """
  Avalia o agente por ``n_eval_episodes`` episódios e retorna a recompensa média e o desvio padrão da recompensa.
  :param env: O ambiente de avaliação
  :param max_steps: Número máximo de passos por episódio
  :param n_eval_episodes: Número de episódios para avaliar o agente
  :param Q: A Q-table
  :param seed: O array de sementes de avaliação (para o taxi-v4)
  """
  episode_rewards = []
  for episode in tqdm(range(n_eval_episodes)):
    if seed:
      state, info = env.reset(seed=seed[episode])
    else:
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(max_steps):
      # Tomar a ação (índice) que tem a recompensa futura esperada máxima dado esse estado
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

## Avaliar nosso agente de Q-Learning 📈

- Normalmente, você deve obter uma recompensa média de 1.0
- O **ambiente é relativamente fácil**, já que o espaço de estados é muito pequeno (16). O que você pode tentar fazer é [substituí-lo pela versão escorregadia (slippery)](https://gymnasium.farama.org/environments/toy_text/frozen_lake/), o que introduz estocasticidade, tornando o ambiente mais complexo.

In [ ]:
# Avaliar nosso Agente
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

In [ ]:
def record_video(env, Qtable, out_directory, fps=1):
  """
  Gera um vídeo de replay do agente
  :param env
  :param Qtable: Qtable do nosso agente
  :param out_directory
  :param fps: quantos quadros por segundo (com o taxi-v4 e o frozenlake-v1 usamos 1)
  """
  images = []
  terminated = False
  truncated = False
  state, info = env.reset(seed=random.randint(0,500))
  img = env.render()
  images.append(img)
  while not terminated or truncated:
    # Tomar a ação (índice) que tem a recompensa futura esperada máxima dado esse estado
    action = np.argmax(Qtable[state][:])
    state, reward, terminated, truncated, info = env.step(action) # Colocamos diretamente next_state = state para a lógica de gravação
    img = env.render()
    images.append(img)
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
# Gravar um vídeo
video_path = "replay.mp4"
record_video(env, Qtable_frozenlake, video_path, video_fps=1)

# Parte 2: Taxi-v4 🚖

## Criar e entender o [Taxi-v4 🚕](https://gymnasium.farama.org/environments/toy_text/taxi/)
---

💡 Um bom hábito ao começar a usar um ambiente é verificar sua documentação

👉 https://gymnasium.farama.org/environments/toy_text/taxi/

---

No `Taxi-v4` 🚕, existem quatro locais designados no mundo em grade indicados por R(ed), G(reen), Y(ellow) e B(lue).

Quando o episódio começa, **o táxi começa em um quadrado aleatório** e o passageiro está em um local aleatório. O táxi vai até o local do passageiro, **pega o passageiro**, vai até o destino do passageiro (outro dos quatro locais especificados) e então **deixa o passageiro (drop-off)**. Uma vez que o passageiro é deixado, o episódio termina.


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/taxi.png" alt="Taxi">


In [ ]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

Existem **500 estados discretos, pois há 25 posições do táxi, 5 locais possíveis do passageiro** (incluindo o caso em que o passageiro está no táxi) e **4 locais de destino.**


In [ ]:
state_space = env.observation_space.n
print("Existem ", state_space, " estados possíveis")

In [ ]:
action_space = env.action_space.n
print("Existem ", action_space, " ações possíveis")

O espaço de ação (o conjunto de ações possíveis que o agente pode tomar) é discreto com **6 ações disponíveis 🎮**:

- 0: mover para o sul (move south)
- 1: mover para o norte (move north)
- 2: mover para o leste (move east)
- 3: mover para o oeste (move west)
- 4: pegar passageiro (pickup passenger)
- 5: deixar passageiro (drop off passenger)

Função de recompensa 💰:

- -1 por passo, a menos que outra recompensa seja acionada.
- +20 por entregar o passageiro.
- -10 por executar as ações de “pegar” (pickup) e “deixar” (drop-off) ilegalmente.

In [ ]:
# Criar nossa Q table com state_size linhas e action_size colunas (500x6)
Qtable_taxi = initialize_q_table(state_space, action_space)
print(Qtable_taxi)
print("Formato da Q-table: ", Qtable_taxi .shape)

## Definir os hiperparâmetros ⚙️



In [ ]:
# Parâmetros de treinamento
n_training_episodes = 25000   # Total de episódios de treinamento
learning_rate = 0.7           # Taxa de aprendizado

# Parâmetros de avaliação
n_eval_episodes = 100        # Número total de episódios de teste

# NÃO MODIFIQUE O EVAL_SEED
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,
 161,131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,145,152,97,106,55,31,85,38,
 112,102,168,123,97,21,83,158,26,80,63,5,81,32,11,28,148] # Semente de avaliação, isso garante que todos os agentes dos colegas sejam treinados na mesma posição inicial do táxi
                                                          # Cada semente tem um estado inicial específico

# Parâmetros do ambiente
env_id = "Taxi-v4"           # Nome do ambiente
max_steps = 99               # Máximo de passos por episódio
gamma = 0.95                 # Taxa de desconto

# Parâmetros de exploração
max_epsilon = 1.0             # Probabilidade de exploração no início
min_epsilon = 0.05           # Probabilidade mínima de exploração
decay_rate = 0.005            # Taxa de decaimento exponencial para a probabilidade de exploração


## Treinar nosso agente de Q-Learning 🏃

In [ ]:
Qtable_taxi = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_taxi)
Qtable_taxi

## Evalie nosso agente de Q-Learning 🏃

## Alguns desafios adicionais 🏆

Parabéns 🥳, você acabou de implementar, treinar e enviar seu primeiro agente de Aprendizagem por Reforço.

Andar no gelo e dirigir táxis está muito chato para você? Tente **mudar o ambiente**, por que não usar a versão escorregadia do FrozenLake-v1? Verifique como eles funcionam [usando a documentação do gymnasium](https://gymnasium.farama.org/) e divirta-se 🎉.



Compreender o Q-Learning é um **passo importante para entender os métodos baseados em valor.**

Na próxima Unidade com Deep Q-Learning, veremos que, embora criar e atualizar uma Q-table tenha sido uma boa estratégia — **no entanto, ela não é escalável.**

Por exemplo, imagine que você cria um agente que aprende a jogar Doom.

<img src="https://vizdoom.cs.put.edu.pl/user/pages/01.tutorial/basic.png" alt="Doom"/>

Doom é um ambiente grande com um enorme espaço de estados (milhões de estados diferentes). Criar e atualizar uma Q-table para esse ambiente não seria eficiente.

É por isso que estudaremos o Deep Q-Learning na próxima unidade, um algoritmo **onde usamos uma rede neural que aproxima, dado um estado, os diferentes valores Q para cada ação.**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/atari-envs.gif" alt="Environments"/>


## Continue aprendendo, continue incrível 🤗